In [7]:
#Install  : pip install "qiskit>=2.1" qiskit-aer matplotlib

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from math import pi, floor, asin, sqrt
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import grover_operator
from qiskit_aer import AerSimulator

# Experiment parameters
N_QUBITS = 4                             # 4 qubits
TARGETS  = ["0011", "1010", "1101"]      # Marked bitstrings
SHOTS    = 8192
OUT_DPI  = 300

# 1. Phase Oracle
def build_phase_oracle(n: int, targets: list) -> QuantumCircuit:
    oracle = QuantumCircuit(n, name=r"$\mathcal{O}_f$")
    for t in targets:
        # Qubits where the target bit is '0' get an X-gate (open control)
        flip = [j for j in range(n) if t[n - 1 - j] == "0"]
        if flip:
            oracle.x(flip)

        # MCZ via H – MCX – H sandwich on the last qubit
        oracle.h(n - 1)
        oracle.mcx(list(range(n - 1)), n - 1)
        oracle.h(n - 1)

        # Undo open controls
        if flip:
            oracle.x(flip)
        oracle.barrier()

    return oracle

# 2. Optimal Iteration Count
def optimal_iterations(n: int, m: int) -> int:
    return max(1, floor(pi / (4.0 * asin(sqrt(m / 2 ** n)))))

# 3. Full Grover Circuit
def build_grover_circuit(n: int, oracle: QuantumCircuit, k: int) -> QuantumCircuit:
    qc = QuantumCircuit(n, n, name="Grover Circuit")
    # Step 1 – uniform superposition
    qc.h(range(n))
    qc.barrier()
    # Step 2 – k Grover iterations
    G = grover_operator(oracle, insert_barriers=True)
    G.name = r"$\mathcal{Q}$"
    for _ in range(k):
        qc.compose(G, inplace=True)
        qc.barrier()
    # Step 3 – measure all qubits
    qc.measure(range(n), range(n))
    return qc

# 4. Simulation on AER simulator
def run_simulation(qc: QuantumCircuit) -> dict:
    sim = AerSimulator(method="statevector")
    tqc = transpile(qc, sim, optimization_level=1)
    return sim.run(tqc, shots=SHOTS).result().get_counts()

# 5. Theoretical Quantities
def theory(n: int, m: int, k: int) -> dict:
    N     = 2 ** n
    theta = asin(sqrt(m / N))          # rotation angle in Hilbert space
    # Success prob = sin²( (2k+1)·θ )  over all M targets
    p_total   = np.sin((2 * k + 1) * theta) ** 2
    p_per_tgt = p_total / m
    # Classical expected oracle calls for random search (without replacement)
    q_classic = (N + 1) / (m + 1)
    # Quantum oracle calls = k
    speedup   = q_classic / k
    return dict(theta=theta, p_total=p_total, p_per_tgt=p_per_tgt,
                q_classic=q_classic, speedup=speedup)

# 6. Figure 1 - Circuit Diagram
def save_circuit_figure(qc: QuantumCircuit, n: int, m: int, k: int,
                        path: str = "grover_circuit.jpg"):

    # Decompose one layer to expose oracle + diffusion labels
    qc_d = qc.decompose(gates_to_decompose=[r"$\mathcal{Q}$"])

    fig = qc_d.draw(
        output="mpl",
        fold=60,
        style={
            "backgroundcolor": "#ffffff",
            "fontsize": 9,
            "displaytext": {r"$\mathcal{O}_f$": r"$\mathcal{O}_f$"},
        },
    )

    fig.suptitle(
        rf"Grover's Algorithm  —  $n={n}$ qubits,  $N=2^{{{n}}}={2**n}$ states,"
        rf"  $M={m}$ targets,  $k={k}$ iteration",
        fontsize=12, fontweight="bold", y=1.03,
    )
    fig.savefig(path, dpi=OUT_DPI, bbox_inches="tight", format="jpeg")
    plt.close(fig)
    print(f"  [saved] {path}")

# 7. Figure 2 – Histogram and Quantum Advantage
def save_results_figure(counts: dict, n: int, targets: list, k: int,
                        th: dict, path: str = "grover_histogram.jpg"):
    N       = 2 ** n
    labels  = [format(i, f"0{n}b") for i in range(N)]
    probs   = np.array([counts.get(s, 0) / SHOTS for s in labels])
    colors  = ["#c0392b" if s in targets else "#2980b9" for s in labels]

    fig, (ax1, ax2) = plt.subplots(
        1, 2, figsize=(14, 5.5),
        gridspec_kw={"width_ratios": [2.8, 1]},
    )
    fig.suptitle(
        "Grover's Algorithm: Measurement Outcomes & Quantum Advantage",
        fontsize=13, fontweight="bold", y=1.01,
    )

    ax1.bar(range(N), probs, color=colors, edgecolor="black", linewidth=0.5, width=0.75)

    uniform = 1.0 / N
    ax1.axhline(uniform, color="#7f7f7f", linestyle="--", linewidth=1.4, zorder=3,
                label=f"Uniform baseline  $1/N = {uniform:.4f}$")
    ax1.axhline(th["p_per_tgt"], color="#8e44ad", linestyle=":", linewidth=1.8, zorder=3,
                label=rf"Theory per target  $\approx {th['p_per_tgt']:.4f}$")

    ax1.set_xticks(range(N))
    ax1.set_xticklabels(labels, rotation=90, fontsize=8, fontfamily="monospace")
    ax1.set_xlabel(r"Measurement Outcome (bitstring  $q_{n-1}\cdots q_0$)", fontsize=11)
    ax1.set_ylabel("Probability", fontsize=11)
    ax1.set_title(rf"Simulation: {SHOTS:,} shots,  $k={k}$ Grover iteration", fontsize=11)
    ax1.set_xlim(-0.6, N - 0.4)
    ax1.set_ylim(0, max(probs) * 1.28)
    ax1.yaxis.grid(True, linestyle="--", alpha=0.4, zorder=0)
    ax1.set_axisbelow(True)

    legend_handles = [
        mpatches.Patch(color="#c0392b", label="Target states (marked)"),
        mpatches.Patch(color="#2980b9", label="Non-target states"),
        plt.Line2D([], [], color="#7f7f7f", linestyle="--",
                   label=f"Uniform $1/N = {uniform:.4f}$"),
        plt.Line2D([], [], color="#8e44ad", linestyle=":",
                   label=rf"Theory per target $\approx {th['p_per_tgt']:.4f}$"),
    ]
    ax1.legend(handles=legend_handles, fontsize=9, loc="upper left",
               framealpha=0.9, edgecolor="grey")
    for s in targets:
        idx = int(s, 2)
        ax1.annotate(
            f"{probs[idx]:.3f}",
            xy=(idx, probs[idx]),
            xytext=(0, 4), textcoords="offset points",
            ha="center", va="bottom", fontsize=8, color="#c0392b", fontweight="bold",
        )
    cat_labels = [f"Classical\nRandom\nO(N/M)", f"Grover\nO(√(N/M))"]
    cat_vals   = [th["q_classic"], float(k)]
    cat_colors = ["#e67e22", "#27ae60"]

    bars = ax2.bar(cat_labels, cat_vals, color=cat_colors, edgecolor="black",
                   linewidth=0.8, width=0.5)
    for bar, val in zip(bars, cat_vals):
        ax2.text(bar.get_x() + bar.get_width() / 2.0, bar.get_height() + 0.08,
                 f"{val:.2f}", ha="center", va="bottom",
                 fontsize=13, fontweight="bold")

    ax2.set_ylabel("Expected Oracle Queries\nto Find One Solution", fontsize=10)
    ax2.set_title(rf"Quantum Advantage  ×{th['speedup']:.1f}", fontsize=12,
                  fontweight="bold", color="#27ae60")
    ax2.set_ylim(0, th["q_classic"] * 1.55)
    ax2.yaxis.grid(True, linestyle="--", alpha=0.4)
    ax2.set_axisbelow(True)
    ax2.text(
        0.5, 0.88,
        rf"$N={2**n}$,  $M={len(targets)}$" + "\n"
        rf"Classical: $(N+1)/(M+1) \approx {th['q_classic']:.2f}$" + "\n"
        rf"Grover: $k={k}$  (1 oracle call)",
        transform=ax2.transAxes, ha="center", va="top", fontsize=8.5,
        bbox=dict(boxstyle="round,pad=0.45", facecolor="#f4f4f4", edgecolor="#aaaaaa"),
    )

    fig.tight_layout()
    fig.savefig(path, dpi=OUT_DPI, bbox_inches="tight", format="jpeg")
    plt.close(fig)
    print(f"  [saved] {path}")


# 8. Console Report
def print_report(n: int, targets: list, k: int, counts: dict, th: dict):
    N        = 2 ** n
    hit      = sum(counts.get(t, 0) for t in targets)
    p_sim    = hit / SHOTS
    th_angle = th["theta"]
    print("\nResults Summary\n")

    print(f"  Qubits n        : {n}")
    print(f"  Search space N  : {N}")
    print(f"  Marked states M : {len(targets)}  {targets}")
    print(f"  Grover angle θ  : {th_angle:.6f} rad")
    print(f"  Optimal iter. k : {k} \n")

    print(f"  Theory  P(success) = sin²((2k+1)θ) = {th['p_total']:.5f}")
    print(f"  Sim.    P(success)                 = {p_sim:.5f}  ({SHOTS:,} shots)")
    print(f"  Theory  P(per target)              = {th['p_per_tgt']:.5f}")

    print("\n  Per-target counts:")
    for t in targets:
        c = counts.get(t, 0)
        print(f"    |{t}⟩  {c:5d} shots  ({c/SHOTS:.4f})")

    print(f"\n  Classical queries : {th['q_classic']:.2f}")
    print(f"  Grover oracle calls        : {k}")
    print(f"  Quantum speedup            : ×{th['speedup']:.2f}")


# Main
def main():
    n, m = N_QUBITS, len(TARGETS)
    k    = optimal_iterations(n, m)
    th   = theory(n, m, k)

    print(f"\nBuilding oracle for targets {TARGETS}  (n={n}, N={2**n}, M={m}, k={k})")
    oracle = build_phase_oracle(n, TARGETS)
    print("Building full Grover circuit")
    qc = build_grover_circuit(n, oracle, k)
    print(f"  Gate count  : {sum(qc.count_ops().values())}")
    print(f"  Circuit depth: {qc.depth()}")

    print("Running simulation on AerSimulator")
    counts = run_simulation(qc)
    print_report(n, TARGETS, k, counts, th)

    print("\nOutput figures have been saved in your current working directory.")
    save_circuit_figure(qc, n, m, k, path="grover_circuit.jpg")
    save_results_figure(counts, n, TARGETS, k, th, path="grover_histogram.jpg")

if __name__ == "__main__":
    main()




Building oracle for targets ['0011', '1010', '1101']  (n=4, N=16, M=3, k=1)
Building full Grover circuit
  Gate count  : 54
  Circuit depth: 20
Running simulation on AerSimulator

Results Summary

  Qubits n        : 4
  Search space N  : 16
  Marked states M : 3  ['0011', '1010', '1101']
  Grover angle θ  : 0.447832 rad
  Optimal iter. k : 1 

  Theory  P(success) = sin²((2k+1)θ) = 0.94922
  Sim.    P(success)                 = 0.94482  (8,192 shots)
  Theory  P(per target)              = 0.31641

  Per-target counts:
    |0011⟩   2597 shots  (0.3170)
    |1010⟩   2624 shots  (0.3203)
    |1101⟩   2519 shots  (0.3075)

  Classical queries : 4.25
  Grover oracle calls        : 1
  Quantum speedup            : ×4.25

Output figures have been saved in your current working directory.
  [saved] grover_circuit.jpg
  [saved] grover_histogram.jpg
